# 📖 Notebook 1: Routing & Load Balancing

This notebook teaches you the **core job** of an API Gateway: routing requests to the right backend service and balancing traffic across multiple instances.

We'll go through three approaches:
- 🚫 **BAD**: Clients call each service directly (tight coupling)
- ✅ **BETTER**: A basic reverse proxy (single entry point)
- 🏆 **BEST**: Full API gateway with path-based routing, load balancing, and health checks

## Learning Objectives

By the end of this notebook, you'll understand:
- Why direct client-to-service calls are problematic
- What a reverse proxy does and why it helps
- How path-based routing directs traffic to different services
- How load balancing distributes requests across service instances
- How health checks keep the system reliable

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 05-microservices/api-gateway
docker compose up -d --build
```

This starts:
- **2 User Service instances** (ports 5001, 5003) — for load balancing
- **1 Order Service instance** (port 5002)
- **nginx API Gateway** (port 8080) — the single entry point
- **Redis** (port 6380) — used in Notebook 2

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import requests
import json

# Helper to pretty-print JSON responses
def show(response):
    """Print HTTP status and JSON body in a readable format."""
    print(f"Status: {response.status_code}")
    try:
        print(json.dumps(response.json(), indent=2))
    except Exception:
        print(response.text)

# Quick health check — make sure everything is running
try:
    r = requests.get("http://localhost:5001/health", timeout=3)
    print(f"✅ User Service 1: {r.json()['status']}")
except Exception as e:
    print(f"❌ User Service 1 not running: {e}")

try:
    r = requests.get("http://localhost:5003/health", timeout=3)
    print(f"✅ User Service 2: {r.json()['status']}")
except Exception as e:
    print(f"❌ User Service 2 not running: {e}")

try:
    r = requests.get("http://localhost:5002/health", timeout=3)
    print(f"✅ Order Service:  {r.json()['status']}")
except Exception as e:
    print(f"❌ Order Service not running: {e}")

try:
    r = requests.get("http://localhost:8080/health", timeout=3)
    print(f"✅ API Gateway:    {r.json()['status']}")
except Exception as e:
    print(f"❌ API Gateway not running: {e}")
    print("   Run: cd 05-microservices/api-gateway && docker compose up -d --build")

---

## 🚫 BAD: Direct Client-to-Service Calls

Without an API gateway, clients must talk to each service directly. This means:
- The client needs to know **every service's address and port**
- If a service moves or scales, **every client must be updated**
- There's **no centralized place** for auth, rate limiting, or logging

```
┌──────────┐     ┌───────────────────┐
│  Client   │────▶│ User Service :5001 │
│           │     └───────────────────┘
│           │     ┌───────────────────┐
│           │────▶│ User Service :5003 │   ← Client must know BOTH instances!
│           │     └───────────────────┘
│           │     ┌───────────────────┐
│           │────▶│ Order Service:5002 │
└──────────┘     └───────────────────┘
```

Let's see what this looks like in code:

In [ ]:
# BAD: The client needs to know each service's address and port.
# If you add a new service or move one to a different server,
# you'd have to update EVERY client.

# Client configuration — imagine maintaining this for 50 services!
SERVICE_REGISTRY = {
    "user_service_1": "http://localhost:5001",
    "user_service_2": "http://localhost:5003",
    "order_service":  "http://localhost:5002",
}

print("🚫 BAD: Client calls each service directly")
print("=" * 55)
print()

# Call the user service directly
print("📡 Calling User Service (instance 1) at port 5001:")
r = requests.get(f"{SERVICE_REGISTRY['user_service_1']}/users/1")
show(r)
print()

# Call the order service directly
print("📡 Calling Order Service at port 5002:")
r = requests.get(f"{SERVICE_REGISTRY['order_service']}/orders/101")
show(r)

In [ ]:
# The client also has to implement its own load balancing!
# This is complex, error-prone, and duplicated across every client.

import random

# Seeded so this cell prints the same split every run -- the lesson here is
# "the client is guessing", not "look, random numbers".
random.seed(7)

def bad_client_side_load_balance(service_urls: list, path: str):
    """BAD: Client picks a random instance and hopes it's healthy."""
    url = random.choice(service_urls)
    return requests.get(f"{url}{path}")

user_service_urls = [
    "http://localhost:5001",
    "http://localhost:5003",
]

print("🚫 BAD: Client-side load balancing")
print("The client randomly picks an instance — no health checking!")
print()

for i in range(6):
    r = bad_client_side_load_balance(user_service_urls, "/users")
    data = r.json()
    print(f"  Request {i+1}: routed to {data['served_by']}")

print()
print("⚠️  Problems with this approach:")
print("   1. Client needs to know ALL service addresses")
print("   2. Client must implement load balancing logic")
print("   3. No health checking — might send traffic to dead instances")
print("   4. Every client (web, mobile, CLI) duplicates this logic")

---

## ✅ BETTER: Basic Reverse Proxy

A **reverse proxy** sits between clients and backend services. Clients send all requests to **one address**, and the proxy forwards them to the right place.

```
┌──────────┐     ┌───────────────┐     ┌───────────────────┐
│  Client   │────▶│ Reverse Proxy │────▶│ Backend Service    │
│           │     │  (nginx:8080) │     │                   │
└──────────┘     └───────────────┘     └───────────────────┘
```

This is already a **huge improvement**:
- Clients only need to know **one address** (the proxy)
- Backend services can move or scale without client changes
- You have a centralized place for logging and monitoring

A simple nginx reverse proxy config looks like this:

```nginx
# BETTER: Simple reverse proxy — one entry point, but no smart routing
server {
    listen 80;
    
    # Everything goes to one backend
    location / {
        proxy_pass http://backend-service:5000;
    }
}
```

But this only forwards to **one service**. What if you have many services?  
That's where an API gateway's **path-based routing** comes in.

In [ ]:
# BETTER: Client only needs one address — the gateway
# Compare this to the BAD approach above where we needed 3 different URLs

GATEWAY = "http://localhost:8080"

print("✅ BETTER: All requests go through the gateway")
print(f"   Client only knows: {GATEWAY}")
print()

# Even though we have 3 service instances behind the scenes,
# the client just calls the gateway
print("📡 Getting users through the gateway:")
r = requests.get(f"{GATEWAY}/api/users/1")
show(r)
print()

print("📡 Getting orders through the gateway:")
r = requests.get(f"{GATEWAY}/api/orders/101")
show(r)
print()

print("💡 Notice: The client doesn't know (or care) that users and orders")
print("   are served by completely different services on different ports!")

---

## 🏆 BEST: API Gateway with Path-Based Routing, Load Balancing & Health Checks

Our nginx API gateway does three things at once:

### 1. Path-Based Routing
The gateway reads the URL path and routes to the correct backend:

```
/api/users/*  ──▶  User Service   (upstream: user-service-1, user-service-2)
/api/orders/* ──▶  Order Service  (upstream: order-service)
```

Here's the relevant nginx config:

```nginx
# Route /api/users to the user service backend
location /api/users {
    proxy_pass http://user_backend/users;
}

# Route /api/orders to the order service backend
location /api/orders {
    proxy_pass http://order_backend/orders;
}
```

### 2. Load Balancing
The `user_backend` upstream has two servers — nginx distributes requests between them:

```nginx
upstream user_backend {
    server user-service-1:5000;   # Instance 1
    server user-service-2:5000;   # Instance 2
}
```

Round-robin here is **deterministic, not random**: nginx keeps a rotation cursor
and hands out peers strictly in turn, so an even number of requests splits exactly
in half. (Our `nginx.conf` sets `worker_processes 1` so that rotation lives in one
place. With multiple workers each keeps its own cursor unless the upstream declares
a shared-memory `zone` — the comment at the top of `nginx.conf` explains why.)

### 3. Health Checks — read the fine print

Each service exposes a `/health` endpoint, but be precise about who calls it:

| Who | What it actually does |
|-----|----------------------|
| `docker-compose` (`depends_on: condition: service_healthy`) | Polls `/health` **at startup only**, so the gateway doesn't boot before its backends |
| nginx **OSS** (this lab) | Never calls `/health`. It only notices failures *passively*, from real traffic: `max_fails=3 fail_timeout=30s` ejects an instance after 3 failed requests, for 30s |
| nginx **Plus** / Envoy / Istio | Real **active** health checks — a background prober calls `/health` on a timer and ejects instances before any user request fails |

So in this lab an instance is only ejected *after* some real requests have
already failed against it. That's the honest trade-off of passive failover,
and it's why Notebook 4 contrasts it with a true circuit breaker.

Let's see all of this in action!

In [ ]:
# BEST: Path-based routing — one URL, multiple services

GATEWAY = "http://localhost:8080"

print("🏆 BEST: Path-Based Routing")
print("=" * 55)
print()
print("The gateway reads the URL path and routes to the correct service.")
print("The client doesn't know which service handles which path.")
print()

# All these go through the SAME gateway on port 8080
routes = [
    ("/api/users",      "→ routed to User Service"),
    ("/api/users/1",    "→ routed to User Service"),
    ("/api/orders",     "→ routed to Order Service"),
    ("/api/orders/101", "→ routed to Order Service"),
]

for path, description in routes:
    r = requests.get(f"{GATEWAY}{path}")
    data = r.json()
    served = data.get("served_by", "unknown")
    print(f"  GET {path:<20} {description}  (served by: {served})")

print()
print("💡 Same gateway URL, different backend services!")
print("   Adding a new service is just a new nginx location block.")

In [ ]:
# BEST: Load Balancing - requests are distributed across instances

import time

print("BEST: Load Balancing")
print("=" * 55)
print()
print("nginx uses round-robin by default: it alternates between instances.")
print("Watch the 'served_by' field change between requests!")
print()

N_REQUESTS = 10
instance_count = {}
rate_limited = 0

for i in range(N_REQUESTS):
    # Gateway rate limiter may return 429 (no JSON body) if we are too fast
    # after previous cells. Back off and retry -- with a few attempts, not one,
    # so a single unlucky retry can't crash the cell on r.json().
    for attempt in range(4):
        r = requests.get(f"{GATEWAY}/api/users")
        if r.status_code != 429:
            break
        rate_limited += 1
        time.sleep(1)
    assert r.status_code == 200, f"gateway returned {r.status_code}: {r.text[:120]}"
    served_by = r.json()["served_by"]
    instance_count[served_by] = instance_count.get(served_by, 0) + 1
    print(f"  Request {i+1:>2}: handled by {served_by}")
    time.sleep(0.25)  # stay under the 5 req/sec gateway limit

print()
print("Distribution:")
for instance, count in sorted(instance_count.items()):
    bar = "#" * count
    print(f"  {instance}: {count} requests  {bar}")
if rate_limited:
    print(f"(Note: {rate_limited} request(s) were briefly rate-limited and retried.)")

# Round-robin is DETERMINISTIC, not random: with two equally-weighted
# instances and a single nginx worker, an even request count must split
# exactly in half. Fail loudly if this lab stops demonstrating that.
assert set(instance_count) == {"user-service-1", "user-service-2"}, (
    f"expected both instances in rotation, got {instance_count}")
assert instance_count["user-service-1"] == instance_count["user-service-2"] == N_REQUESTS // 2, (
    f"round-robin should split {N_REQUESTS} requests exactly in half, got {instance_count}")

print()
print("nginx spreads the load evenly - no single instance gets overwhelmed.")
print("Other strategies: 'least_conn' (least busy), 'ip_hash' (sticky sessions).")


In [ ]:
# BEST: Health Checks — the gateway knows which backends are alive

print("🏆 BEST: Health Checks")
print("=" * 55)
print()

# Check gateway health
r = requests.get(f"{GATEWAY}/health")
print("Gateway health:")
show(r)
print()

# Check individual service health (through direct ports)
services = [
    ("User Service 1", "http://localhost:5001/health"),
    ("User Service 2", "http://localhost:5003/health"),
    ("Order Service",  "http://localhost:5002/health"),
]

print("Backend service health:")
for name, url in services:
    try:
        r = requests.get(url, timeout=2)
        status = r.json()["status"]
        instance = r.json()["instance"]
        print(f"  ✅ {name}: {status} (instance {instance})")
    except Exception as e:
        print(f"  ❌ {name}: DOWN ({e})")

print()
print("💡 What nginx OSS actually does when user-service-1 dies:")
print("   1. The NEXT request routed to it fails (or times out after 2s)")
print("   2. proxy_next_upstream retries that request on user-service-2,")
print("      so the client usually still gets a 200 -- just slower")
print("   3. After max_fails=3 such failures, nginx stops sending it traffic")
print("      for fail_timeout=30s, then quietly tries it again")
print()
print("⚠️  Note what is NOT happening: nginx OSS never calls /health on its own.")
print("   Failures are discovered from real user traffic, so the first few")
print("   requests after a crash DO pay the timeout. Active health checking")
print("   needs nginx Plus, Envoy, or a service mesh (see Notebook 4).")

In [ ]:
# Let's compare: BAD (direct) vs BEST (gateway) side by side

import time

print("Comparison: Direct Calls vs API Gateway")
print("=" * 60)
print()

# BAD: Direct calls
print("BAD: Direct calls (client manages service discovery)")
print("   URLs the client needs to know: 3")
print("   - http://localhost:5001 (user instance 1)")
print("   - http://localhost:5003 (user instance 2)")
print("   - http://localhost:5002 (order service)")
print("   Load balancing: client-side (manual)")
print("   Auth/Rate limiting: each service implements its own")
print()

# BEST: Gateway
print("BEST: API Gateway (centralized control)")
print("   URLs the client needs to know: 1")
print("   - http://localhost:8080")
print("   Load balancing: handled by gateway (automatic)")
print("   Auth/Rate limiting: centralized at gateway")
print()

def measure_avg_latency(url, n=10, throttle=0.25):
    """Time n requests. Sleep between them so we stay under the 5 req/sec gateway limit."""
    times = []
    for _ in range(n):
        start = time.time()
        requests.get(url)
        times.append((time.time() - start) * 1000)
        time.sleep(throttle)
    return sum(times) / len(times)

direct_ms = measure_avg_latency("http://localhost:5001/users")
gateway_ms = measure_avg_latency("http://localhost:8080/api/users")

overhead_ms = gateway_ms - direct_ms

print("Latency (avg of 10 requests, throttled to respect rate limit):")
print(f"   Direct call:      {direct_ms:.1f} ms")
print(f"   Through gateway:  {gateway_ms:.1f} ms")
print(f"   Overhead:         ~{overhead_ms:.1f} ms")
print()
print("Read that overhead number honestly. It is NOT nginx's cost -- nginx's own")
print("proxy overhead is tens of microseconds. What you are measuring here is one")
print("extra hop plus a fresh TCP connection through Docker's port forwarding,")
print("which on macOS/Windows dominates everything else. On a real deployment,")
print("where the gateway and the services share a network, budget ~1-2ms.")
print()
print("The point stands either way: one extra hop buys you routing, load")
print("balancing, auth, rate limiting, and observability in one place.")

# The gateway must add ONE hop, not multiply latency. If this ever fails,
# something is wrong (retrying upstreams, DNS per request, a stuck backend).
assert overhead_ms < direct_ms + 25, (
    f"gateway hop should cost about one extra round-trip, but overhead was "
    f"{overhead_ms:.1f} ms on top of a {direct_ms:.1f} ms direct call")


## 🧭 How Routing Works Under the Hood

When a request arrives at the gateway, nginx follows this process:

```
1. Client sends:  GET http://localhost:8080/api/users/1
                              │
2. nginx matches:  location /api/users  ← longest prefix match
                              │
3. nginx rewrites: /api/users/1  →  /users/1
                              │
4. nginx picks:    user-service-1:5000 (round-robin)
                              │
5. nginx forwards: GET http://user-service-1:5000/users/1
                              │
6. Backend responds → nginx returns response to client
```

The key config line is:
```nginx
location /api/users {
    proxy_pass http://user_backend/users;
}
```

- `location /api/users` — matches any URL starting with `/api/users`
- `proxy_pass http://user_backend/users` — forwards to the upstream, replacing `/api/users` with `/users`

## 📚 Summary

### What We Learned

| Approach | Entry Points | Load Balancing | Health Checks | Centralized Control |
|----------|-------------|----------------|---------------|--------------------|
| 🚫 BAD (direct) | Many (one per service) | Client-side | None | ❌ |
| ✅ BETTER (reverse proxy) | One | None | None | Partial |
| 🏆 BEST (API gateway) | One | Automatic | Passive ejection (nginx OSS) / active probes (nginx Plus, Envoy) | ✅ |

### Key Takeaways

1. **Single entry point** — clients only need one URL, not one per service
2. **Path-based routing** — the gateway maps URL paths to backend services
3. **Load balancing** — traffic is automatically spread across service instances
4. **Health checks** — nginx OSS ejects a backend only *after* real requests to it fail (`max_fails`/`fail_timeout`); active `/health` probing needs nginx Plus, Envoy, or a service mesh
5. **Minimal overhead** — the gateway costs one extra network hop (~1-2ms in a real deployment; more in this lab, because Docker Desktop's port forwarding dominates the measurement)

### Interview Tip

> When drawing a system design, just say: *"I'll add an API Gateway to handle routing and basic middleware"* and draw a single box. Don't over-explain — the gateway is important but not the interesting part of most designs.

### Next Up

In **Notebook 2**, we'll add **rate limiting** and **API key authentication** at the gateway level — protecting all your services from one place.